# Superstore Sales Analysis
### Using Subqueries · CTEs · Window Functions
---
**Objective:** Analyze sales data using SQL by applying Subqueries, CTEs, and Window Functions to solve business queries.

 
| Step | Description |
|------|-------------|
| 1 | Load dataset → `superstore_raw` |
| 2 | Create normalized tables (`customers`, `orders`, `products`) |
| 3 | Subqueries — filter above-average sales, highest order per customer |
| 4 | CTEs — total sales aggregations |
| 5 | Window Functions — `ROW_NUMBER`, `RANK` |
| 6 | Combined query — JOIN + CTE + Window Function |
| 7 | Business Insights — top / bottom customers, single-order, above-average |

## Setup & Imports

In [ ]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_colwidth", 40)

print("Libraries ready!")

✅ Libraries ready!


## Step 1 — Load Data into `superstore_raw`

In [2]:
# Load CSV
df = pd.read_csv("Sample_-_Superstore.csv", encoding="latin1")

# Normalise column names
df.columns = [c.strip().replace(" ", "_").replace("-", "_") for c in df.columns]

# Connect to in-memory SQLite
conn = sqlite3.connect(":memory:")

# Load raw table
df.to_sql("superstore_raw", conn, index=False, if_exists="replace")

n = pd.read_sql("SELECT COUNT(*) AS cnt FROM superstore_raw", conn)["cnt"][0]
print("Rows loaded into superstore_raw:", n)
print("Columns:", df.columns.tolist())
pd.read_sql("SELECT * FROM superstore_raw LIMIT 5", conn)

Rows loaded into superstore_raw: 9994
Columns: ['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,Hon Deluxe Fabric Upholstered Stacki...,731.94,3,0.00,219.58
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typ...,14.62,2,0.00,6.87
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangu...,957.58,5,0.45,-383.03
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52


## Step 2 — Create Normalized Tables
Three child tables are built from `superstore_raw` using `SELECT DISTINCT`.

```
superstore_raw
   ├── customers  (Customer_ID, Customer_Name, Segment, City, State, Region)
   ├── orders     (Order_ID, Order_Date, Ship_Date, Ship_Mode, Customer_ID, Sales, Qty, Discount, Profit)
   └── products   (Product_ID, Product_Name, Category, Sub_Category)
```

In [3]:
# ── customers ──────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE customers AS
    SELECT DISTINCT
        Customer_ID, Customer_Name, Segment,
        Country, City, State, Region
    FROM superstore_raw
""")

# ── orders ─────────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE orders AS
    SELECT DISTINCT
        Order_ID, Order_Date, Ship_Date, Ship_Mode,
        Customer_ID, Sales, Quantity, Discount, Profit
    FROM superstore_raw
""")

# ── products ───────────────────────────────────────────────────────
conn.execute("""
    CREATE TABLE products AS
    SELECT DISTINCT
        Product_ID, Product_Name, Category, Sub_Category
    FROM superstore_raw
""")

conn.commit()

for t in ["superstore_raw", "customers", "orders", "products"]:
    cnt = pd.read_sql("SELECT COUNT(*) AS c FROM " + t, conn)["c"][0]
    print("  " + t + " →", cnt, "rows")

  superstore_raw → 9994 rows
  customers → 4688 rows
  orders → 9993 rows
  products → 1894 rows


In [4]:
print("Sample — customers"); display(pd.read_sql("SELECT * FROM customers LIMIT 4", conn))
print("\nSample — orders");   display(pd.read_sql("SELECT * FROM orders   LIMIT 4", conn))
print("\nSample — products"); display(pd.read_sql("SELECT * FROM products LIMIT 4", conn))

Sample — customers


,Customer_ID,Customer_Name,Segment,Country,City,State,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West



Sample — orders


,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,261.96,2,0.00,41.91
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,731.94,3,0.00,219.58
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,14.62,2,0.00,6.87
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,957.58,5,0.45,-383.03



Sample — products


,Product_ID,Product_Name,Category,Sub_Category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,Hon Deluxe Fabric Upholstered Stacki...,Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typ...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangu...,Furniture,Tables


## Step 3 — Subqueries

### 3-A · Orders where Sales > Average Sales

In [5]:
q3a = """
SELECT
    o.Order_ID,
    c.Customer_Name,
    ROUND(o.Sales, 2)                           AS Sales,
    ROUND((SELECT AVG(Sales) FROM orders), 2)   AS Avg_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales > (SELECT AVG(Sales) FROM orders)
ORDER BY o.Sales DESC
LIMIT 12
"""
r3a = pd.read_sql(q3a, conn)
avg_s = pd.read_sql("SELECT ROUND(AVG(Sales),2) AS a FROM orders", conn)["a"][0]
above = pd.read_sql("SELECT COUNT(*) AS c FROM orders WHERE Sales > (SELECT AVG(Sales) FROM orders)", conn)["c"][0]
print("Overall average sales : $", avg_s)
print("Orders above average  :", above, "of", pd.read_sql("SELECT COUNT(*) AS c FROM orders",conn)["c"][0])
r3a

Overall average sales : $ 229.85
Orders above average  : 2359 of 9993


,Order_ID,Customer_Name,Sales,Avg_Sales
0,CA-2014-145317,Sean Miller,22638.48,229.85
1,CA-2014-145317,Sean Miller,22638.48,229.85
2,CA-2014-145317,Sean Miller,22638.48,229.85
3,CA-2014-145317,Sean Miller,22638.48,229.85
4,CA-2014-145317,Sean Miller,22638.48,229.85
5,CA-2016-118689,Tamara Chand,17499.95,229.85
6,CA-2016-118689,Tamara Chand,17499.95,229.85
7,CA-2016-118689,Tamara Chand,17499.95,229.85
8,CA-2016-118689,Tamara Chand,17499.95,229.85
9,CA-2016-118689,Tamara Chand,17499.95,229.85


### 3-B · Highest-Sales Order per Customer (Correlated Subquery)

In [6]:
q3b = """
SELECT
    c.Customer_Name,
    o.Order_ID,
    ROUND(o.Sales, 2) AS Peak_Sales
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales = (
    SELECT MAX(o2.Sales)
    FROM   orders o2
    WHERE  o2.Customer_ID = o.Customer_ID
)
ORDER BY Peak_Sales DESC
LIMIT 12
"""
r3b = pd.read_sql(q3b, conn)
print("Highest single-order sales per customer (top 12):")
r3b

Highest single-order sales per customer (top 12):


,Customer_Name,Order_ID,Peak_Sales
0,Sean Miller,CA-2014-145317,22638.48
1,Sean Miller,CA-2014-145317,22638.48
2,Sean Miller,CA-2014-145317,22638.48
3,Sean Miller,CA-2014-145317,22638.48
4,Sean Miller,CA-2014-145317,22638.48
5,Tamara Chand,CA-2016-118689,17499.95
6,Tamara Chand,CA-2016-118689,17499.95
7,Tamara Chand,CA-2016-118689,17499.95
8,Tamara Chand,CA-2016-118689,17499.95
9,Tamara Chand,CA-2016-118689,17499.95


## Step 4 — CTEs

### 4-A · Total Sales per Customer

In [7]:
q4a = """
WITH customer_sales AS (
    SELECT
        o.Customer_ID,
        c.Customer_Name,
        ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT * FROM customer_sales
ORDER  BY Total_Sales DESC
LIMIT  12
"""
r4a = pd.read_sql(q4a, conn)
print("Total sales per customer (top 12):")
r4a

Total sales per customer (top 12):


,Customer_ID,Customer_Name,Total_Sales
0,KL-16645,Ken Lonsdale,141752.29
1,SE-20110,Sanjit Engle,134303.82
2,AB-10105,Adrian Barton,130262.14
3,SM-20320,Sean Miller,125215.25
4,CL-12565,Clay Ludtke,119686.01
5,SC-20095,Sanjit Chand,113138.67
6,SV-20365,Seth Vernon,103238.55
7,ZC-21910,Zuschuss Carroll,96308.48
8,ME-17320,Maria Etezadi,95973.55
9,LA-16780,Laura Armstrong,95405.44


### 4-B · Customers whose Total Sales exceed the Customer Average (CTE + Subquery)

In [8]:
q4b = """
WITH customer_sales AS (
    SELECT
        o.Customer_ID,
        c.Customer_Name,
        ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
SELECT *
FROM   customer_sales
WHERE  Total_Sales > (SELECT AVG(Total_Sales) FROM customer_sales)
ORDER  BY Total_Sales DESC
"""
r4b = pd.read_sql(q4b, conn)
print("Customers above average total sales:", len(r4b))
r4b.head(12)

Customers above average total sales: 278


,Customer_ID,Customer_Name,Total_Sales
0,KL-16645,Ken Lonsdale,141752.29
1,SE-20110,Sanjit Engle,134303.82
2,AB-10105,Adrian Barton,130262.14
3,SM-20320,Sean Miller,125215.25
4,CL-12565,Clay Ludtke,119686.01
5,SC-20095,Sanjit Chand,113138.67
6,SV-20365,Seth Vernon,103238.55
7,ZC-21910,Zuschuss Carroll,96308.48
8,ME-17320,Maria Etezadi,95973.55
9,LA-16780,Laura Armstrong,95405.44


## Step 5 — Window Functions

### 5-A · ROW_NUMBER — Order sequence within each customer (PARTITION BY Customer)

In [9]:
q5a = """
SELECT
    c.Customer_Name,
    o.Order_ID,
    o.Order_Date,
    ROUND(o.Sales, 2) AS Sales,
    ROW_NUMBER() OVER (
        PARTITION BY o.Customer_ID
        ORDER BY     o.Order_Date
    ) AS Order_Seq
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
ORDER BY c.Customer_Name, Order_Seq
LIMIT 15
"""
r5a = pd.read_sql(q5a, conn)
print("ROW_NUMBER() per customer — first 15 rows:")
r5a

ROW_NUMBER() per customer — first 15 rows:


,Customer_Name,Order_ID,Order_Date,Sales,Order_Seq
0,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,1
1,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,2
2,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,3
3,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,4
4,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,5
5,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,6
6,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,7
7,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,8
8,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,9
9,Aaron Bergman,CA-2014-156587,3/7/2014,48.71,10


### 5-B · RANK — Customers ranked by Total Sales

In [10]:
q5b = """
SELECT
    Customer_Name,
    Total_Sales,
    RANK()       OVER (ORDER BY Total_Sales DESC) AS Rank,
    DENSE_RANK() OVER (ORDER BY Total_Sales DESC) AS Dense_Rank
FROM (
    SELECT o.Customer_ID, c.Customer_Name,
           ROUND(SUM(o.Sales), 2) AS Total_Sales
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID
    GROUP  BY o.Customer_ID, c.Customer_Name
)
ORDER BY Rank
LIMIT 15
"""
r5b = pd.read_sql(q5b, conn)
print("RANK vs DENSE_RANK by total sales (top 15):")
r5b

RANK vs DENSE_RANK by total sales (top 15):


,Customer_Name,Total_Sales,Rank,Dense_Rank
0,Ken Lonsdale,141752.29,1,1
1,Sanjit Engle,134303.82,2,2
2,Adrian Barton,130262.14,3,3
3,Sean Miller,125215.25,4,4
4,Clay Ludtke,119686.01,5,5
5,Sanjit Chand,113138.67,6,6
6,Seth Vernon,103238.55,7,7
7,Zuschuss Carroll,96308.48,8,8
8,Maria Etezadi,95973.55,9,9
9,Laura Armstrong,95405.44,10,10


## Step 6 — Final Combined Query
**JOIN + CTE + Window Function** → Customer · Total Sales · Rank

In [11]:
q6 = """
WITH customer_totals AS (
    -- CTE: aggregate sales per customer
    SELECT
        o.Customer_ID,
        c.Customer_Name,
        c.Segment,
        c.Region,
        ROUND(SUM(o.Sales),   2) AS Total_Sales,
        ROUND(SUM(o.Profit),  2) AS Total_Profit,
        COUNT(DISTINCT o.Order_ID)  AS Order_Count
    FROM   orders o
    JOIN   customers c ON o.Customer_ID = c.Customer_ID  -- JOIN
    GROUP  BY o.Customer_ID, c.Customer_Name, c.Segment, c.Region
),
ranked AS (
    -- Window Function: rank inside each segment
    SELECT
        Customer_Name,
        Segment,
        Region,
        Total_Sales,
        Total_Profit,
        Order_Count,
        RANK() OVER (ORDER BY Total_Sales DESC)                    AS Overall_Rank,
        RANK() OVER (PARTITION BY Segment ORDER BY Total_Sales DESC) AS Segment_Rank
    FROM customer_totals
)
SELECT *
FROM   ranked
ORDER  BY Overall_Rank
LIMIT  20
"""
r6 = pd.read_sql(q6, conn)
print("Final combined result — Customer | Total Sales | Profit | Orders | Overall & Segment Rank (top 20):")
r6

Final combined result — Customer | Total Sales | Profit | Orders | Overall & Segment Rank (top 20):


,Customer_Name,Segment,Region,Total_Sales,Total_Profit,Order_Count,Overall_Rank,Segment_Rank
0,Sanjit Engle,Consumer,West,85466.07,18554.74,11,1,1
1,Adrian Barton,Consumer,Central,72367.85,27224.03,10,2,2
2,Ken Lonsdale,Consumer,West,70876.15,4034.27,12,3,3
3,Seth Vernon,Consumer,East,68825.70,7196.55,10,4,4
4,Sanjit Chand,Consumer,West,56569.34,23029.65,9,5,5
5,Sean Miller,Home Office,South,50086.10,-3961.48,5,6,1
6,Caroline Jumper,Consumer,East,44659.90,3434.97,8,7,6
7,Clay Ludtke,Consumer,East,43522.18,7735.13,12,8,7
8,Clay Ludtke,Consumer,West,43522.18,7735.13,12,8,7
9,Laura Armstrong,Corporate,Central,43366.11,10295.60,11,10,1


## Step 7 — Business Insights

### 7-1 · Top 5 Customers

In [12]:
q_top5 = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, c.Segment,
           ROUND(SUM(o.Sales),2) AS Total_Sales,
           COUNT(DISTINCT o.Order_ID) AS Orders
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name, c.Segment
)
SELECT Customer_Name, Segment, Total_Sales, Orders,
       RANK() OVER (ORDER BY Total_Sales DESC) AS Rank
FROM cs
ORDER BY Total_Sales DESC LIMIT 5
"""
top5 = pd.read_sql(q_top5, conn)
print("Top 5 Customers by Total Sales:")
display(top5)
print("Insight: Top customer drove $", top5["Total_Sales"].max(), "in sales.")

Top 5 Customers by Total Sales:


,Customer_Name,Segment,Total_Sales,Orders,Rank
0,Ken Lonsdale,Consumer,141752.29,12,1
1,Sanjit Engle,Consumer,134303.82,11,2
2,Adrian Barton,Consumer,130262.14,10,3
3,Sean Miller,Home Office,125215.25,5,4
4,Clay Ludtke,Consumer,119686.01,12,5


Insight: Top customer drove $ 141752.29 in sales.


### 7-2 · Bottom 5 Customers

In [13]:
q_bot5 = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, c.Segment,
           ROUND(SUM(o.Sales),2) AS Total_Sales,
           COUNT(DISTINCT o.Order_ID) AS Orders
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name, c.Segment
)
SELECT Customer_Name, Segment, Total_Sales, Orders,
       RANK() OVER (ORDER BY Total_Sales ASC) AS Rank_From_Bottom
FROM cs
ORDER BY Total_Sales ASC LIMIT 5
"""
bot5 = pd.read_sql(q_bot5, conn)
print("Bottom 5 Customers by Total Sales:")
display(bot5)
print("Insight: Lowest customer had only $", bot5["Total_Sales"].min(), "in total sales.")

Bottom 5 Customers by Total Sales:


,Customer_Name,Segment,Total_Sales,Orders,Rank_From_Bottom
0,Lela Donovan,Corporate,5.30,1,1
1,Thais Sissman,Consumer,9.67,2,2
2,Carl Jackson,Corporate,16.52,1,3
3,Mitch Gastineau,Corporate,16.74,1,4
4,Roy Skaria,Home Office,44.66,2,5


Insight: Lowest customer had only $ 5.3 in total sales.


### 7-3 · Customers Who Placed Only One Order

In [14]:
q_one = """
SELECT c.Customer_Name, c.Segment, c.Region,
       COUNT(DISTINCT o.Order_ID)  AS Order_Count,
       ROUND(SUM(o.Sales), 2)      AS Total_Sales
FROM   orders o
JOIN   customers c ON o.Customer_ID = c.Customer_ID
GROUP  BY o.Customer_ID, c.Customer_Name, c.Segment, c.Region
HAVING COUNT(DISTINCT o.Order_ID) = 1
ORDER  BY Total_Sales DESC
"""
r_one = pd.read_sql(q_one, conn)
print("Customers with only 1 order:", len(r_one))
display(r_one.head(10))
print("Insight:", len(r_one), "customers placed just one order — potential churn risk.")

Customers with only 1 order: 12


,Customer_Name,Segment,Region,Order_Count,Total_Sales
0,Jenna Caffey,Consumer,West,1,1058.11
1,Susan MacKendrick,Consumer,East,1,1043.04
2,Theresa Coyne,Corporate,East,1,1038.26
3,Jocasta Rupert,Consumer,South,1,863.88
4,Patricia Hirasaki,Home Office,South,1,729.65
5,Anthony O'Donnell,Corporate,West,1,161.28
6,Roland Murray,Consumer,East,1,98.35
7,Anemone Ratner,Consumer,South,1,88.15
8,Ricardo Emerson,Consumer,East,1,48.36
9,Mitch Gastineau,Corporate,South,1,16.74


Insight: 12 customers placed just one order — potential churn risk.


### 7-4 · Customers with Above-Average Total Sales

In [15]:
q_abv = """
WITH cs AS (
    SELECT o.Customer_ID, c.Customer_Name, c.Segment,
           ROUND(SUM(o.Sales),2) AS Total_Sales
    FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
    GROUP BY o.Customer_ID, c.Customer_Name, c.Segment
)
SELECT Customer_Name, Segment, Total_Sales
FROM   cs
WHERE  Total_Sales > (SELECT AVG(Total_Sales) FROM cs)
ORDER  BY Total_Sales DESC
"""
r_abv = pd.read_sql(q_abv, conn)
avg_c = pd.read_sql("WITH cs AS (SELECT SUM(Sales) AS s FROM orders GROUP BY Customer_ID) SELECT ROUND(AVG(s),2) AS a FROM cs", conn)["a"][0]
print("Average customer total sales : $", avg_c)
print("Customers above average      :", len(r_abv))
display(r_abv.head(10))
print("Insight:", len(r_abv), "customers account for the bulk of revenue.")

Average customer total sales : $ 2896.49
Customers above average      : 278


,Customer_Name,Segment,Total_Sales
0,Ken Lonsdale,Consumer,141752.29
1,Sanjit Engle,Consumer,134303.82
2,Adrian Barton,Consumer,130262.14
3,Sean Miller,Home Office,125215.25
4,Clay Ludtke,Consumer,119686.01
5,Sanjit Chand,Consumer,113138.67
6,Seth Vernon,Consumer,103238.55
7,Zuschuss Carroll,Consumer,96308.48
8,Maria Etezadi,Home Office,95973.55
9,Laura Armstrong,Corporate,95405.44


Insight: 278 customers account for the bulk of revenue.


### 7-5 · Highest Single-Order Value per Customer

In [16]:
q_max = """
SELECT
    c.Customer_Name,
    o.Order_ID,
    o.Order_Date,
    ROUND(o.Sales, 2)   AS Peak_Order_Sales,
    ROUND(o.Profit, 2)  AS Profit_On_Peak
FROM orders o
JOIN customers c ON o.Customer_ID = c.Customer_ID
WHERE o.Sales = (
    SELECT MAX(o2.Sales) FROM orders o2
    WHERE  o2.Customer_ID = o.Customer_ID
)
ORDER BY Peak_Order_Sales DESC
LIMIT 10
"""
r_max = pd.read_sql(q_max, conn)
print("Highest order value per customer (top 10):")
display(r_max)
print("Insight: Single largest order was $", r_max["Peak_Order_Sales"].max())

Highest order value per customer (top 10):


,Customer_Name,Order_ID,Order_Date,Peak_Order_Sales,Profit_On_Peak
0,Sean Miller,CA-2014-145317,3/18/2014,22638.48,-1811.08
1,Sean Miller,CA-2014-145317,3/18/2014,22638.48,-1811.08
2,Sean Miller,CA-2014-145317,3/18/2014,22638.48,-1811.08
3,Sean Miller,CA-2014-145317,3/18/2014,22638.48,-1811.08
4,Sean Miller,CA-2014-145317,3/18/2014,22638.48,-1811.08
5,Tamara Chand,CA-2016-118689,10/2/2016,17499.95,8399.98
6,Tamara Chand,CA-2016-118689,10/2/2016,17499.95,8399.98
7,Tamara Chand,CA-2016-118689,10/2/2016,17499.95,8399.98
8,Tamara Chand,CA-2016-118689,10/2/2016,17499.95,8399.98
9,Tamara Chand,CA-2016-118689,10/2/2016,17499.95,8399.98


Insight: Single largest order was $ 22638.48


---
## Summary of Techniques Used

| SQL Concept | Applied In |
|---|---|
| **Subquery** | Step 3-A (avg filter), Step 3-B (correlated max), Step 7-5 |
| **CTE** | Step 4-A, 4-B, 6, 7-1 through 7-4 |
| **RANK()** | Step 5-B, Step 6 (overall & segment), Step 7-1, 7-2 |
| **DENSE_RANK()** | Step 5-B |
| **ROW_NUMBER() + PARTITION BY** | Step 5-A |
| **JOIN** | Steps 3-B, 4-A, 4-B, 5-A, 5-B, 6, all Step 7 |
| **HAVING** | Step 7-3 |

## Key Business Insights

1. A small group of customers (above-average segment) drives most revenue — prioritise retention.
2. Single-order customers represent a churn risk — targeted re-engagement campaigns recommended.
3. The bottom 5 customers have very low sales; check if they are loss-making after discounts.
4. Peak single-order values reveal high-value transaction opportunities per customer.
5. Segment-level ranking (Step 6) helps identify which customers lead within Consumer / Corporate / Home Office.